In [1]:
import pandas as pd 

X_testnormal = pd.read_csv("X_testnormal.csv")
X_trainnormal = pd.read_csv("X_trainnormal.csv")
y_trainnormal = pd.read_csv("y_trainnormal.csv")
test_ids = pd.read_csv("test_ids.csv")

In [2]:
import optuna 

from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split


from sklearn.metrics import make_scorer, mean_squared_error
import numpy as np

def log_rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

log_rmse_scorer = make_scorer(
    log_rmse,
    greater_is_better=False
)



In [5]:
y_trainnormallog = np.log1p(y_trainnormal)

In [6]:
from sklearn.model_selection import cross_val_score, KFold
from lightgbm import LGBMRegressor
import optuna

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

def objective(trial):

    n_estimators = trial.suggest_int("n_estimators", 600, 1500)

    learning_rate = trial.suggest_float(
        "learning_rate",
        0.005,
        0.1,
        log=True
    )

    num_leaves = trial.suggest_int(
        "num_leaves",
        15,
        100
    )

    max_depth = trial.suggest_int(
        "max_depth",
        3,
        10
    )

    min_child_samples = trial.suggest_int(
        "min_child_samples",
        5,
        50
    )

    subsample = trial.suggest_float(
        "subsample",
        0.5,
        1.0
    )

    colsample_bytree = trial.suggest_float(
        "colsample_bytree",
        0.5,
        1.0
    )

    reg_alpha = trial.suggest_float(
        "reg_alpha",
        1e-4,
        10,
        log=True
    )

    reg_lambda = trial.suggest_float(
        "reg_lambda",
        1e-4,
        100,
        log=True
    )

    model = LGBMRegressor(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        num_leaves=num_leaves,
        max_depth=max_depth,
        min_child_samples=min_child_samples,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,
        random_state=42,
        verbosity=-1
    )

    score = cross_val_score(
        model,
        X_trainnormal,
        y_trainnormallog,
        cv=cv,
        scoring="neg_root_mean_squared_error"
    )

    return -score.mean()


study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=100)

print("Best CV score:", study.best_value)
print("Best parameters:", study.best_params)

[I 2026-08-30 00:53:42,721] A new study created in memory with name: no-name-6eedb27b-4602-47df-a7fd-0504e5a37efe
[I 2026-08-30 00:53:52,263] Trial 0 finished with value: 0.13092018074921835 and parameters: {'n_estimators': 1468, 'learning_rate': 0.013563511501029704, 'num_leaves': 32, 'max_depth': 4, 'min_child_samples': 49, 'subsample': 0.6258622569486945, 'colsample_bytree': 0.6592565975942168, 'reg_alpha': 0.6415365880994073, 'reg_lambda': 0.7591907619829624}. Best is trial 0 with value: 0.13092018074921835.
[I 2026-08-30 00:53:59,104] Trial 1 finished with value: 0.13448826808701853 and parameters: {'n_estimators': 1146, 'learning_rate': 0.03529047453094981, 'num_leaves': 94, 'max_depth': 6, 'min_child_samples': 39, 'subsample': 0.8932645372448683, 'colsample_bytree': 0.8841767059649603, 'reg_alpha': 1.6398934444232705, 'reg_lambda': 92.20406003474794}. Best is trial 0 with value: 0.13092018074921835.
[I 2026-08-30 00:54:05,672] Trial 2 finished with value: 0.129660182539522 and p

Best CV score: 0.128138544042172
Best parameters: {'n_estimators': 1205, 'learning_rate': 0.011631025341630335, 'num_leaves': 83, 'max_depth': 6, 'min_child_samples': 25, 'subsample': 0.70213039759884, 'colsample_bytree': 0.515111327104394, 'reg_alpha': 0.005686510163442302, 'reg_lambda': 1.7937158431298093}


In [ ]:
LGB_model = XGBRegressor(
    **study.best_params,
    random_state = 42
)

LGB_model.fit(X_trainnormal,y_trainnormal)

,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.5166727377596607
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [ ]:
preds = LGB_model.predict(X_testnormal)
preds = preds.squeeze()



In [ ]:
test_ids = test_ids.squeeze()
df = pd.DataFrame({"Id":test_ids.values, "SalePrice" : preds })

df.to_csv("submissionLGB_model.csv", index = False)